# Bài toán:
> **Phân cụm dữ liệu bằng Hierarchical Clustering – Phân cụm phân cấp (unsupervised learning)**

Mục tiêu
- Xây dựng mô hình phân cụm phân cấp với thư viện sklearn (`AgglomerativeClustering`, kiểu liên kết **ward**).
- Vẽ và đọc **dendrogram** (cây phân cấp).
- Chọn số cụm bằng chỉ số **Silhouette**.
- Đánh giá kết quả bằng chỉ số **ARI** (so với nhãn thật).

Dữ liệu
- Bài toán 1: Dữ liệu tự sinh bằng `make_blobs`. Mỗi dữ liệu là một điểm trên mặt phẳng Oxy.
- Bài toán 2: Bộ dữ liệu **Wine** có sẵn trong sklearn: 178 mẫu rượu vang, mỗi mẫu có 13 chỉ số hoá học. Có sẵn nhãn giống nho (3 loại) – **không dùng nhãn để huấn luyện**, chỉ dùng để chấm điểm kết quả.
- Bài tập: áp dụng cùng các bước cho các bộ dữ liệu khác (Breast Cancer, Iris).

Mô hình hóa bài toán: bài toán phân cụm.

- Đầu vào: $N$ điểm dữ liệu, mỗi điểm có $D$ thuộc tính.
- Đầu ra: Chỉ số cụm của mỗi điểm dữ liệu.

Ý tưởng của phân cụm phân cấp (gộp dần):
1. Ban đầu mỗi điểm là một cụm riêng.
2. Tìm **hai cụm gần nhau nhất** và gộp lại thành một cụm.
3. Lặp lại bước 2 cho tới khi chỉ còn 1 cụm.

Quá trình gộp tạo thành một cây (dendrogram). Muốn có $k$ cụm thì "cắt" cây ở mức phù hợp.

**Ward** là cách đo khoảng cách giữa hai cụm: gộp hai cụm nào làm các cụm **ít bị phân tán nhất**. Ward cho ra các cụm tròn, kích thước đều – giống K-means.

# Các bước làm

## Các thư viện sử dụng

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs, load_wine, load_breast_cancer, load_iris
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score

from scipy.cluster.hierarchy import linkage, dendrogram

## Bài toán 1: Mô phỏng phân cụm phân cấp với dữ liệu tự sinh (không gian Oxy)

### Chuẩn bị dữ liệu
- Sinh ngẫu nhiên n_samples = 30 điểm, xung quanh 3 tâm cố định
    - random_state: cố định hàm random – để mỗi lần chạy sinh ra cùng một bộ điểm
- Mỗi điểm dữ liệu có 2 chiều
- `y` là nhãn thật (điểm được sinh ra từ tâm nào) – chỉ dùng để đánh giá

In [ ]:
n_samples = 30
random_state = 42
center_points = [[1, 1], [-1, -1], [1, -1]]

X, y = make_blobs(n_samples=n_samples, centers=center_points,
                  cluster_std=0.4, random_state=random_state)
print("Số chiều dữ liệu: ", X.shape, y.shape)
print("5 điểm dữ liệu đầu tiên: \n", X[:5])

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(X[:, 0], X[:, 1], c='blue')
plt.title("Các điểm dữ liệu trước khi phân cụm. Số lượng: {}".format(n_samples))
plt.show()

### Vẽ dendrogram (cây phân cấp)

- `linkage(X, method='ward')`: thực hiện quá trình gộp dần, ghi lại thứ tự gộp.
- `dendrogram(Z)`: vẽ cây.

**Cách đọc dendrogram:**
- Trục hoành: các điểm dữ liệu.
- Trục tung: khoảng cách lúc gộp. Hai nhánh nối nhau càng cao thì hai cụm đó càng khác nhau.
- Kẻ một đường ngang cắt qua cây: **số nhánh bị cắt = số cụm**.

In [ ]:
Z = linkage(X, method='ward')

plt.figure(figsize=(12, 5))
dendrogram(Z)
plt.axhline(y=3, c='red', ls='--', label='Đường cắt → 3 cụm')
plt.title("Dendrogram (ward)")
plt.xlabel("Chỉ số điểm dữ liệu")
plt.ylabel("Khoảng cách gộp")
plt.legend()
plt.show()

**Nhận xét:** ba nhánh lớn nối với nhau ở độ cao rất lớn (khoảng 6 và 8), trong khi bên dưới các điểm được gộp ở độ cao nhỏ (dưới 2). Đường cắt ở độ cao 3 cắt qua 3 nhánh → dữ liệu có 3 cụm.

### Dựng mô hình và huấn luyện
- Sử dụng `AgglomerativeClustering` của sklearn, xem chi tiết tại [tài liệu hướng dẫn](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.AgglomerativeClustering.html)
- Hai tham số chính:
    - `n_clusters`: số cụm muốn lấy
    - `linkage='ward'`: kiểu liên kết ward (đây cũng là giá trị mặc định)
- `fit_predict(X)`: huấn luyện và trả về nhãn cụm của từng điểm.

Lưu ý: khác với K-means, mô hình này **không có tâm cụm** và **không có hàm `predict()`** cho điểm mới.

In [ ]:
k_cluster = 3
model = AgglomerativeClustering(n_clusters=k_cluster, linkage='ward')
y_pred = model.fit_predict(X)

print("Nhãn cụm của 10 điểm đầu tiên:", y_pred[:10])
print("Số điểm trong mỗi cụm:", np.bincount(y_pred))

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(X[:, 0], X[:, 1], c=y_pred)
plt.title("Các điểm dữ liệu sau khi phân cụm (k = {})".format(k_cluster))
plt.show()

### Đánh giá mô hình

**1. Silhouette – không cần nhãn thật**

Với mỗi điểm, Silhouette so sánh:
- $a$: khoảng cách trung bình đến các điểm **cùng cụm**
- $b$: khoảng cách trung bình đến các điểm của **cụm gần nhất khác**

$$s = \frac{b - a}{\max(a, b)}$$

- $s$ gần **1**: điểm nằm đúng cụm, các cụm tách biệt rõ.
- $s$ gần **0**: điểm nằm ở ranh giới giữa hai cụm.
- $s$ **âm**: điểm có lẽ bị xếp nhầm cụm.

`silhouette_score(X, y_pred)` trả về trung bình $s$ của tất cả các điểm. **Càng cao càng tốt.**

**2. ARI (Adjusted Rand Index) – cần nhãn thật**

ARI đo mức độ giống nhau giữa cách chia cụm của mô hình và nhãn thật:
- ARI = **1**: khớp hoàn toàn.
- ARI ≈ **0**: giống như chia ngẫu nhiên.

ARI **không quan tâm tên nhãn**: mô hình gọi một cụm là "0" còn nhãn thật gọi là "2" cũng không sao, chỉ cần các điểm được chia nhóm giống nhau.

`adjusted_rand_score(y, y_pred)` – thứ tự: nhãn thật trước, nhãn dự đoán sau.

In [ ]:
sil = silhouette_score(X, y_pred)
ari = adjusted_rand_score(y, y_pred)

print("Silhouette:", round(sil, 3))
print("ARI       :", round(ari, 3))

### Chọn số cụm bằng Silhouette

Trong thực tế ta **không biết** có bao nhiêu cụm. Cách làm:
- Thử nhiều giá trị $k$ (ví dụ từ 2 đến 6).
- Với mỗi $k$, huấn luyện mô hình và tính Silhouette.
- Chọn $k$ có Silhouette **cao nhất**.

Lưu ý: Silhouette chỉ tính được khi $k \geq 2$.

In [ ]:
k_values = [2, 3, 4, 5, 6]
sil_values = []

for k in k_values:
    model_k = AgglomerativeClustering(n_clusters=k, linkage='ward')
    labels_k = model_k.fit_predict(X)
    sil_values.append(silhouette_score(X, labels_k))
    print("k = {}: Silhouette = {:.3f}".format(k, sil_values[-1]))

plt.figure(figsize=(6, 4))
plt.plot(k_values, sil_values, 'o-')
plt.xlabel("Số cụm k")
plt.ylabel("Silhouette")
plt.title("Chọn số cụm bằng Silhouette")
plt.show()

best_k = k_values[np.argmax(sil_values)]
print("Số cụm tốt nhất theo Silhouette:", best_k)

### Bài tập 1
Yêu cầu: Sinh dữ liệu có **4 cụm** và làm lại toàn bộ các bước trên.

- Sinh dữ liệu: `X1, y1 = make_blobs(n_samples=40, centers=4, cluster_std=0.6, random_state=7)`
- Vẽ dữ liệu và vẽ dendrogram. Nhìn dendrogram, bạn đoán dữ liệu có mấy cụm?
- Thử $k$ từ 2 đến 6, tính Silhouette, chọn $k$ tốt nhất.
- Huấn luyện mô hình với $k$ tốt nhất, vẽ kết quả, tính ARI.

Gợi ý: copy các ô code ở trên, chỉ đổi tên biến `X`, `y` thành `X1`, `y1`.

Kết quả phải ra được: hình dendrogram, đồ thị Silhouette theo $k$, hình phân cụm và giá trị ARI.

In [ ]:
# Sinh dữ liệu 4 cụm

# Vẽ dữ liệu và dendrogram

# Chọn k bằng Silhouette

# Huấn luyện với k tốt nhất, vẽ kết quả và tính ARI

## Bài toán 2: Ứng dụng phân cụm phân cấp để phân nhóm rượu vang

- Đặt vấn đề:
    - Có kết quả phân tích hoá học của 178 mẫu rượu vang (13 chỉ số: độ cồn, axit, magie, phenol, màu sắc,...).
    - Muốn tự động gom các mẫu có đặc điểm hoá học giống nhau thành nhóm.
- Giải pháp:
    - Dùng phân cụm phân cấp (ward) để chia nhóm.
    - Dùng Silhouette để chọn số nhóm, dùng ARI để kiểm tra với giống nho thật.

Các bước (sẽ dùng lại y hệt cho bài tập 3, 4):
1. Đọc dữ liệu
2. Chuẩn hoá dữ liệu
3. Vẽ dendrogram
4. Chọn số cụm bằng Silhouette
5. Huấn luyện mô hình
6. Đánh giá bằng ARI và trực quan hoá

### Bước 1: Đọc dữ liệu
- Mỗi dòng (một mẫu rượu) là một điểm dữ liệu
- Cột `target` là giống nho thật – **chỉ dùng để đánh giá**, không đưa vào mô hình

In [ ]:
wine = load_wine(as_frame=True)
df = wine.frame
print("Kích thước dữ liệu:", df.shape)
df.head()

In [ ]:
X_wine = df.drop(columns='target').values   # 13 thuộc tính
y_wine = df['target'].values                # nhãn thật, chỉ dùng để đánh giá
print("X_wine:", X_wine.shape, "| y_wine:", y_wine.shape)

### Bước 2: Chuẩn hoá dữ liệu

Các thuộc tính có thang đo rất khác nhau: `proline` có giá trị hàng trăm – hàng nghìn, còn `hue` chỉ khoảng 0.5 – 1.7. Phân cụm dựa trên **khoảng cách**, nên thuộc tính có số lớn sẽ lấn át các thuộc tính khác.

→ Dùng `StandardScaler` đưa mọi thuộc tính về cùng thang đo (trung bình 0, độ lệch chuẩn 1).

In [ ]:
print("Giá trị lớn nhất của từng cột (5 cột cuối):", X_wine.max(axis=0)[-5:])

scaler = StandardScaler()
X_wine_scaled = scaler.fit_transform(X_wine)

print("Sau chuẩn hoá - trung bình 5 cột đầu:", np.round(X_wine_scaled.mean(axis=0)[:5], 2))
print("Sau chuẩn hoá - độ lệch chuẩn 5 cột đầu:", np.round(X_wine_scaled.std(axis=0)[:5], 2))

### Bước 3: Vẽ dendrogram

Dữ liệu có 178 điểm nên cây rất rối. Dùng `truncate_mode='lastp', p=20` để chỉ vẽ 20 nhánh trên cùng.

In [ ]:
Z_wine = linkage(X_wine_scaled, method='ward')

plt.figure(figsize=(12, 5))
dendrogram(Z_wine, truncate_mode='lastp', p=20)
plt.title("Dendrogram rút gọn – Wine (ward)")
plt.ylabel("Khoảng cách gộp")
plt.show()

### Bước 4 + 5: Huấn luyện mô hình

Trước tiên huấn luyện với $k = 3$ (bài tập 2 sẽ dùng Silhouette để kiểm tra lựa chọn này).

In [ ]:
model_wine = AgglomerativeClustering(n_clusters=3, linkage='ward')
y_wine_pred = model_wine.fit_predict(X_wine_scaled)

print("Số mẫu trong mỗi cụm:", np.bincount(y_wine_pred))

### Bước 6: Đánh giá và trực quan hoá

In [ ]:
print("Silhouette:", round(silhouette_score(X_wine_scaled, y_wine_pred), 3))
print("ARI       :", round(adjusted_rand_score(y_wine, y_wine_pred), 3))

**Bảng đối chiếu:** mỗi dòng là một giống nho thật, mỗi cột là một cụm của mô hình. Nếu mỗi dòng chỉ có một ô lớn thì phân cụm tốt.

In [ ]:
pd.crosstab(y_wine, y_wine_pred, rownames=['Giống nho thật'], colnames=['Cụm dự đoán'])

Dữ liệu có 13 chiều nên không vẽ trực tiếp được. Dùng **PCA** thu gọn xuống 2 chiều **chỉ để vẽ hình**.

In [ ]:
X_wine_2d = PCA(n_components=2).fit_transform(X_wine_scaled)

plt.figure(figsize=(12, 5))
plt.subplot(121)
plt.scatter(X_wine_2d[:, 0], X_wine_2d[:, 1], c=y_wine_pred, s=15)
plt.title("Cụm của mô hình")
plt.subplot(122)
plt.scatter(X_wine_2d[:, 0], X_wine_2d[:, 1], c=y_wine, s=15)
plt.title("Giống nho thật")
plt.show()

**Nhận xét:** Silhouette khoảng 0.28 – thấp hơn nhiều so với dữ liệu tự sinh, vì dữ liệu thật có các nhóm nằm sát nhau. Nhưng ARI khoảng 0.79 cho thấy mô hình tìm ra các nhóm khá khớp với giống nho thật.

### Bài tập 2
Yêu cầu: Kiểm tra lựa chọn $k = 3$ cho dữ liệu Wine bằng Silhouette.

- Thử $k$ từ 2 đến 8 trên `X_wine_scaled`, tính Silhouette cho mỗi $k$.
- Vẽ đồ thị Silhouette theo $k$. $k$ nào tốt nhất?
- Tính thêm ARI cho mỗi $k$ (so với `y_wine`). $k$ có Silhouette cao nhất có trùng với $k$ có ARI cao nhất không?

Gợi ý: dùng lại vòng lặp `for k in k_values` ở Bài toán 1.

In [ ]:
# code

### Bài tập 3: Cùng mô hình, dữ liệu khác – Breast Cancer

Bộ dữ liệu **Breast Cancer** (có sẵn trong sklearn): 569 mẫu khối u, mỗi mẫu có 30 chỉ số đo từ ảnh tế bào (bán kính, độ nhám, độ lõm,...). Nhãn thật có 2 loại: lành tính và ác tính.

Yêu cầu: làm lại **đúng 6 bước** của Bài toán 2 cho bộ dữ liệu này.
1. Đọc dữ liệu: `cancer = load_breast_cancer(as_frame=True)`, tách `X_cancer` (bỏ cột `target`) và `y_cancer`.
2. Chuẩn hoá bằng `StandardScaler`.
3. Vẽ dendrogram rút gọn.
4. Thử $k$ từ 2 đến 6, chọn $k$ có Silhouette cao nhất.
5. Huấn luyện `AgglomerativeClustering` với $k$ đó, `linkage='ward'`.
6. Tính ARI, in bảng crosstab, vẽ hình bằng PCA.

Câu hỏi: Silhouette có chọn đúng số loại khối u (2) không? ARI bằng bao nhiêu?

In [ ]:
# Bước 1: Đọc dữ liệu

# Bước 2: Chuẩn hoá

# Bước 3: Dendrogram

# Bước 4: Chọn k bằng Silhouette

# Bước 5: Huấn luyện

# Bước 6: Đánh giá và trực quan hoá

### Bài tập 4: Cùng mô hình, dữ liệu khác – Iris

Bộ dữ liệu **Iris** (có sẵn trong sklearn): 150 bông hoa diên vĩ, mỗi bông có 4 số đo (chiều dài, chiều rộng của đài hoa và cánh hoa). Nhãn thật có 3 loài hoa.

Yêu cầu:
- Làm lại 6 bước như Bài tập 3 với `load_iris(as_frame=True)`.
- Silhouette chọn $k$ bằng bao nhiêu? Có trùng với số loài hoa thật (3) không?
- Huấn luyện thêm mô hình với $k = 3$ và tính ARI. So sánh ARI của $k$ do Silhouette chọn và của $k = 3$.
- Nhìn hình PCA, giải thích vì sao Silhouette lại chọn như vậy.

In [ ]:
# code

## Tổng kết

Cấu trúc chung khi phân cụm một bộ dữ liệu mới bằng phân cụm phân cấp:

| Bước | Việc làm | Hàm |
|---|---|---|
| 1 | Đọc dữ liệu, tách thuộc tính `X` và nhãn thật `y` (nếu có) | `load_...`, `pandas` |
| 2 | Chuẩn hoá dữ liệu | `StandardScaler().fit_transform(X)` |
| 3 | Vẽ dendrogram để quan sát | `linkage(X, method='ward')`, `dendrogram(Z)` |
| 4 | Thử nhiều $k$, chọn $k$ có Silhouette cao nhất | `silhouette_score(X, labels)` |
| 5 | Huấn luyện mô hình | `AgglomerativeClustering(n_clusters=k, linkage='ward').fit_predict(X)` |
| 6 | Đánh giá với nhãn thật (nếu có), trực quan hoá | `adjusted_rand_score(y, labels)`, `PCA` |

Ghi nhớ:
- **Silhouette** (từ −1 đến 1, càng cao càng tốt): không cần nhãn thật, dùng để **chọn số cụm**.
- **ARI** (1 = khớp hoàn toàn, 0 = ngẫu nhiên): cần nhãn thật, dùng để **kiểm tra** kết quả.
- Số cụm "đẹp nhất" theo Silhouette không phải lúc nào cũng trùng với số nhóm thật.